# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as a single object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print("\nDataset License:", metadata.license)
print("Published:", metadata.datePublished)

# Optionally, pretty print keywords and use cases
print("Keywords:", ', '.join(metadata.keywords))
print("Use Cases:")
pprint.pprint(metadata.dataUseCases)

## 2. Data Overview
Review available record sets, fields, and their IDs.
This will help to identify which record sets and fields we may want to explore and reference them by their `@id`.

**Note:** All entities (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
# List all RecordSets using metadata
record_sets = metadata.recordSet  # This is a list

if not record_sets:
    print("No record sets defined directly in metadata. Attempting to auto-detect from mlcroissant Dataset object...")

# Try loading record set IDs directly from the dataset object
record_sets_ids = dataset.record_sets
print("Available Record Sets (@id):")
for rs_id in record_sets_ids:
    print(f"- RecordSet: {rs_id}")

# For each RecordSet, list the available fields and their @id
for rs_id in record_sets_ids:
    print(f"\nFields for RecordSet '@id': {rs_id}")
    rs_obj = dataset.get_record_set(rs_id)
    fields = rs_obj.fields
    for field in fields:
        print(f"  Field: {field['@id']} | name: {field.get('name', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In this dataset, we will load all available record sets, using their `@id`, and create a DataFrame for each.

In [ ]:
# Extract data from each record set
record_sets = list(dataset.record_sets)
print('Record sets to load:', record_sets)
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for RecordSet: {record_set_id}, shape: {dataframes[record_set_id].shape}")
        print("Columns:", dataframes[record_set_id].columns.tolist())
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for RecordSet: {record_set_id}")

# Choose the primary record set for further analysis
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    print(f"\nUsing main RecordSet: {main_record_set_id}")
    print("Fields:", dataframes[main_record_set_id].columns.tolist())
else:
    print("No record set found to use for analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's filter based on a numeric field (e.g., GAD-7 Score) and group by a demographic field (e.g., gender).

In [ ]:
# Choose a numeric field and a group field, referencing by @id
# Example: 'gad7_score', 'gender' (replace with actual column names from the previous section)

df = dataframes.get(main_record_set_id, pd.DataFrame())

# Attempt to auto-detect numeric fields from columns
numeric_fields = [col for col in df.columns if 'score' in col.lower() or 'age' in col.lower() or df[col].dtype in ['int64', 'float64']]
if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Using numeric field '{numeric_field}' for filtering.")
else:
    numeric_field = None
    print("No numeric field detected.")

# Attempt to auto-detect group field (e.g., 'gender', 'level_of_education', etc.)
group_fields = [col for col in df.columns if col.lower() in ['gender', 'level_of_education', 'marital_status']]
if group_fields:
    group_field = group_fields[0]
    print(f"Using group field '{group_field}'.")
else:
    group_field = None

# Filtering and normalizing steps
if numeric_field and not df.empty:
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10

    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by '{group_field}':")
        display(grouped_df.head())
else:
    print("Unable to perform EDA due to missing numeric or group fields.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we'll plot the distribution of the chosen numeric field and, if possible, show grouped averages by the chosen demographic field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution
if numeric_field and not df.empty:
    plt.figure(figsize=(10, 6))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

# Grouped bar plot
if group_field and numeric_field and not df.empty:
    plt.figure(figsize=(8, 6))
    grouped = df.groupby(group_field)[numeric_field].mean().reset_index()
    sns.barplot(x=group_field, y=numeric_field, data=grouped)
    plt.title(f"Average {numeric_field} by {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset provides a rich sample of mental health indicators from Kilifi County, Kenya.
- Key demographic and numeric fields (such as GAD-7 scores) can be analyzed and visualized by using their `@id`.
- Data exploration shows how scores vary across demographic groups, supporting further research into targeted mental health interventions.
- All exploration steps reference data entities by their Croissant `@id`, ensuring correct and reproducible analysis.